# Titled Tuesday — Modeling

Implements every approach recommended in `ANALYSIS.md`, evaluated on the temporal split (**train = Feb 10, test = Mar 10**):

1. **Baselines** — marginal frequencies; Elo-only.
2. **Ordinal (proportional-odds) logistic regression** — custom implementation; the recommended primary model.
3. **Multinomial logistic regression** — ablation for the ordinality assumption.
4. **Gradient-boosted trees** — sklearn `HistGradientBoostingClassifier` (LightGBM-style histogram GBT, native NaN handling, no extra dependency).
5. **Bayesian Bradley–Terry–Davidson** — latent player skills with Glicko-informed priors, MAP estimate with analytic gradients.
6. **Ensemble + temperature scaling** — ordinal LR + GBT averaged, calibrated on round-grouped out-of-fold predictions.

Metrics: multiclass **log loss** (primary), Brier score, accuracy, per-class recall; reliability diagrams at the end.

Label encoding used throughout: `loss=0, draw=1, win=2`; probability columns always in that order. Total runtime ≈ 1–2 minutes.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize, minimize_scalar
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, recall_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

CLASSES = ["loss", "draw", "win"]
LBL = {c: i for i, c in enumerate(CLASSES)}

df = pd.read_parquet("data/processed/games.parquet")
train = df[df["tournament"].str.contains("february")].reset_index(drop=True)
test = df[df["tournament"].str.contains("march")].reset_index(drop=True)
y_tr = train["outcome"].map(LBL).to_numpy()
y_te = test["outcome"].map(LBL).to_numpy()

num_feats = [c for c in df.columns
             if c.startswith(("diff_", "white_", "black_", "mean_", "abs_"))
             and df[c].dtype.kind in "fi" and not c.endswith("_reported")]
X_tr, X_te = train[num_feats], test[num_feats]
print(f"train {len(train)} games | test {len(test)} games | {len(num_feats)} numeric features")

In [ ]:
results, test_probs = {}, {}

def brier(y, P):
    Y = np.eye(3)[y]
    return float(((P - Y) ** 2).sum(axis=1).mean())

def evaluate(name, P, y=y_te):
    P = np.clip(P, 1e-12, None)
    P = P / P.sum(axis=1, keepdims=True)
    pred = P.argmax(axis=1)
    rec = recall_score(y, pred, labels=[0, 1, 2], average=None, zero_division=0)
    results[name] = {
        "log_loss": log_loss(y, P, labels=[0, 1, 2]),
        "brier": brier(y, P),
        "accuracy": float((pred == y).mean()),
        "recall_loss": rec[0], "recall_draw": rec[1], "recall_win": rec[2],
    }
    test_probs[name] = P
    print(f"{name:26s} log loss {results[name]['log_loss']:.4f} | "
          f"brier {results[name]['brier']:.4f} | acc {results[name]['accuracy']:.3f}")
    return P

## 1. Baselines

Any real model must beat the Elo-only baseline; the marginal baseline catches gross miscalibration.

In [ ]:
# Marginal: train class frequencies
p_marg = np.bincount(y_tr, minlength=3) / len(y_tr)
evaluate("Marginal", np.tile(p_marg, (len(test), 1)))

# Elo-only: E = P(win) + 0.5 P(draw), with P(draw) fixed at the train draw rate
p_draw = (y_tr == 1).mean()
E = test["white_elo_expected"].to_numpy()
P_elo = evaluate("Elo-only", np.column_stack([
    np.clip(1 - E - p_draw / 2, 1e-6, None),   # loss
    np.full_like(E, p_draw),                   # draw
    np.clip(E - p_draw / 2, 1e-6, None),       # win
]))

## 2. Ordinal (proportional-odds) logistic regression — primary model

$$P(Y \le k \mid x) = \sigma(\tau_k - \beta^\top x), \qquad \tau_0 < \tau_1$$

Latent game advantage $Z = \beta^\top x + \varepsilon$, $\varepsilon \sim \mathrm{Logistic}$; the draw is the band $(\tau_0, \tau_1]$. Spends $p+2$ parameters vs the multinomial's $2p+2$. Implemented directly (L-BFGS on the penalized NLL, ordering enforced via $\tau_1 = \tau_0 + e^{t}$), wrapped as an sklearn estimator so it composes with imputation/scaling pipelines and CV.

In [ ]:
class OrdinalLogit(BaseEstimator, ClassifierMixin):
    """Proportional-odds logistic regression for 3 ordered classes (0<1<2), L2 penalty."""

    def __init__(self, alpha=1.0):
        self.alpha = alpha

    @staticmethod
    def _probs(z, tau0, tau1):
        c0 = 1.0 / (1.0 + np.exp(-(tau0 - z)))   # P(Y<=0)
        c1 = 1.0 / (1.0 + np.exp(-(tau1 - z)))   # P(Y<=1)
        P = np.column_stack([c0, c1 - c0, 1.0 - c1])
        return np.clip(P, 1e-12, None)

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y)
        n, p = X.shape

        def nll(params):
            beta, tau0, t = params[:p], params[p], params[p + 1]
            P = self._probs(X @ beta, tau0, tau0 + np.exp(t))
            ll = np.log(P[np.arange(n), y]).sum()
            return -(ll - 0.5 * self.alpha * beta @ beta)

        x0 = np.zeros(p + 2)
        x0[p], x0[p + 1] = -0.2, np.log(0.3)
        self.res_ = minimize(nll, x0, method="L-BFGS-B")
        self.beta_ = self.res_.x[:p]
        self.tau_ = (self.res_.x[p], self.res_.x[p] + np.exp(self.res_.x[p + 1]))
        self.classes_ = np.array([0, 1, 2])
        return self

    def predict_proba(self, X):
        z = np.asarray(X, dtype=float) @ self.beta_
        return self._probs(z, *self.tau_)

    def predict(self, X):
        return self.predict_proba(X).argmax(axis=1)


def preprocess():
    return [SimpleImputer(strategy="median", add_indicator=True), StandardScaler()]

ordinal_pipe = make_pipeline(*preprocess(), OrdinalLogit(alpha=10.0))
ordinal_pipe.fit(X_tr, y_tr)
P_ord = evaluate("Ordinal LR", ordinal_pipe.predict_proba(X_te))
print(f"thresholds tau = {np.round(ordinal_pipe[-1].tau_, 3)}")

In [ ]:
# Which features move the latent advantage scale? (top |coefficients|)
imp = ordinal_pipe[0]
feat_names = list(np.asarray(num_feats)) + [f"missing_{num_feats[i]}" for i in imp.indicator_.features_]
coefs = pd.Series(ordinal_pipe[-1].beta_, index=feat_names).sort_values(key=np.abs, ascending=False)
coefs.head(15).round(3)

## 3. Multinomial logistic regression — ordinality ablation

Same preprocessing, $2p+2$ parameters with class-specific slopes. If this does **not** beat the ordinal model, the proportional-odds restriction is empirically supported.

In [ ]:
mnl_pipe = make_pipeline(*preprocess(), LogisticRegression(max_iter=3000, C=0.1))
mnl_pipe.fit(X_tr, y_tr)
P_mnl = evaluate("Multinomial LR", mnl_pipe.predict_proba(X_te))

## 4. Gradient-boosted trees

Histogram GBT with native NaN routing (no imputation needed — missingness stays informative), shallow trees + early stopping against overfitting at $n \approx 2{,}000$. An upper-bound probe on nonlinear signal the linear models miss.

In [ ]:
gbt = HistGradientBoostingClassifier(
    max_depth=3, learning_rate=0.06, max_iter=500,
    l2_regularization=1.0, early_stopping=True, validation_fraction=0.15,
    random_state=0,
)
gbt.fit(X_tr, y_tr)
P_gbt = evaluate("GBT", gbt.predict_proba(X_te))
print(f"trees used: {gbt.n_iter_}")

## 5. Bayesian Bradley–Terry–Davidson with Glicko priors

Each player has latent skill $\theta_i$ with $\pi_i = e^{\theta_i + \gamma\,\mathbb{1}[i=\text{white}]}$ and Davidson draw handling:

$$P(\text{white wins}) = \frac{\pi_w}{D}, \quad P(\text{draw}) = \frac{\nu\sqrt{\pi_w \pi_b}}{D}, \quad D = \pi_w + \pi_b + \nu\sqrt{\pi_w \pi_b}$$

**Priors**: $\theta_i \sim \mathcal{N}(c\,R_i,\ (c\,RD_i)^2)$ with $c = \ln 10 / 400$ — the Glicko rating $R_i$ (pre-game Elo) and deviation $RD_i$ from the stats endpoint are the mean and SD of Chess.com's own skill posterior, so this is measurement-error modeling, not a convenience prior. Weak priors on $\gamma$ (white advantage) and $\log \nu$.

Fitted by **MAP with analytic gradients** (~700 + 2 parameters, L-BFGS, seconds). Players unseen in training keep their prior mean — i.e., they fall back to a calibrated Elo prediction, which is exactly the right behavior. A full posterior (PyMC/NumPyro, non-centered parameterization) is the natural next iteration.

In [ ]:
C = np.log(10) / 400.0

def long_view(d):
    w = d[["white_username", "white_elo", "white_blitz_rd"]].rename(
        columns=lambda c: c.replace("white_", ""))
    b = d[["black_username", "black_elo", "black_blitz_rd"]].rename(
        columns=lambda c: c.replace("black_", ""))
    return pd.concat([w, b], ignore_index=True)

lv_tr, lv_te = long_view(train), long_view(test)
players = sorted(set(lv_tr["username"]) | set(lv_te["username"]))
pidx = {u: i for i, u in enumerate(players)}

# Prior mean: mean pre-game Elo in train; test-only players use their (pre-game) test Elo
elo_prior = (lv_tr.groupby("username")["elo"].mean()
             .combine_first(lv_te.groupby("username")["elo"].mean()))
rd = pd.concat([lv_tr, lv_te]).groupby("username")["blitz_rd"].mean()
elo0 = elo_prior.mean()
mu = C * (elo_prior.reindex(players).to_numpy() - elo0)
sigma = C * np.clip(rd.reindex(players).fillna(75).to_numpy(), 30, 150)

wi_tr = train["white_username"].map(pidx).to_numpy()
bi_tr = train["black_username"].map(pidx).to_numpy()
n_pl = len(players)

def neg_log_post(params):
    theta, gamma, eta = params[:n_pl], params[n_pl], params[n_pl + 1]
    nu = np.exp(eta)
    lw, lb = theta[wi_tr] + gamma, theta[bi_tr]
    pw, pb = np.exp(lw), np.exp(lb)
    s = np.sqrt(pw * pb)
    D = pw + pb + nu * s
    ll = np.where(y_tr == 2, lw, np.where(y_tr == 0, lb, eta + 0.5 * (lw + lb))) - np.log(D)
    lp = (ll.sum()
          - 0.5 * (((theta - mu) / sigma) ** 2).sum()
          - 0.5 * gamma ** 2
          - 0.5 * ((eta - np.log(0.3)) / 2.0) ** 2)

    # analytic gradient
    dlnD_w = (pw + 0.5 * nu * s) / D
    dlnD_b = (pb + 0.5 * nu * s) / D
    g_w = np.where(y_tr == 2, 1.0, np.where(y_tr == 1, 0.5, 0.0)) - dlnD_w
    g_b = np.where(y_tr == 0, 1.0, np.where(y_tr == 1, 0.5, 0.0)) - dlnD_b
    g_theta = np.zeros(n_pl)
    np.add.at(g_theta, wi_tr, g_w)
    np.add.at(g_theta, bi_tr, g_b)
    g_theta -= (theta - mu) / sigma ** 2
    g_gamma = g_w.sum() - gamma
    g_eta = ((y_tr == 1) - nu * s / D).sum() - (eta - np.log(0.3)) / 4.0
    return -lp, -np.concatenate([g_theta, [g_gamma, g_eta]])

x0 = np.concatenate([mu, [0.05, np.log(0.3)]])
opt = minimize(neg_log_post, x0, jac=True, method="L-BFGS-B",
               options={"maxiter": 2000})
theta_hat, gamma_hat, nu_hat = opt.x[:n_pl], opt.x[n_pl], np.exp(opt.x[n_pl + 1])
print(f"converged: {opt.success} | white advantage gamma = {gamma_hat:.4f} "
      f"({gamma_hat / C:.0f} Elo) | draw nu = {nu_hat:.4f}")

def davidson_proba(wi, bi):
    pw = np.exp(theta_hat[wi] + gamma_hat)
    pb = np.exp(theta_hat[bi])
    s = np.sqrt(pw * pb)
    D = pw + pb + nu_hat * s
    return np.column_stack([pb / D, nu_hat * s / D, pw / D])

P_dav = evaluate("Davidson-BT (MAP)", davidson_proba(
    test["white_username"].map(pidx).to_numpy(),
    test["black_username"].map(pidx).to_numpy()))

## 6. Ensemble + temperature scaling

Average the ordinal LR and GBT probabilities, then calibrate with power/temperature scaling $p_k^{1/T} / \sum_j p_j^{1/T}$. $T$ is fitted on **round-grouped out-of-fold** train predictions (GroupKFold by round) so the calibration set respects the dependence structure — never on test.

In [ ]:
gkf = GroupKFold(n_splits=5)
groups = train["round"].to_numpy()
oof = {"ord": np.zeros((len(train), 3)), "gbt": np.zeros((len(train), 3))}

for tr_idx, va_idx in gkf.split(X_tr, y_tr, groups):
    o = make_pipeline(*preprocess(), OrdinalLogit(alpha=10.0)).fit(X_tr.iloc[tr_idx], y_tr[tr_idx])
    g = HistGradientBoostingClassifier(
        max_depth=3, learning_rate=0.06, max_iter=500, l2_regularization=1.0,
        early_stopping=True, validation_fraction=0.15, random_state=0,
    ).fit(X_tr.iloc[tr_idx], y_tr[tr_idx])
    oof["ord"][va_idx] = o.predict_proba(X_tr.iloc[va_idx])
    oof["gbt"][va_idx] = g.predict_proba(X_tr.iloc[va_idx])

P_oof = 0.5 * (oof["ord"] + oof["gbt"])

def temp_loss(T, P, y):
    Q = np.clip(P, 1e-12, None) ** (1.0 / T)
    Q /= Q.sum(axis=1, keepdims=True)
    return log_loss(y, Q, labels=[0, 1, 2])

T_hat = minimize_scalar(temp_loss, bounds=(0.3, 3.0), method="bounded",
                        args=(P_oof, y_tr)).x
print(f"fitted temperature T = {T_hat:.3f} "
      f"(OOF log loss {temp_loss(1.0, P_oof, y_tr):.4f} -> {temp_loss(T_hat, P_oof, y_tr):.4f})")

P_ens = 0.5 * (P_ord + P_gbt)
P_ens = np.clip(P_ens, 1e-12, None) ** (1.0 / T_hat)
P_ens /= P_ens.sum(axis=1, keepdims=True)
P_ens = evaluate("Ensemble + temp", P_ens)

## 7. Results

In [ ]:
res = pd.DataFrame(results).T.sort_values("log_loss")
res.round(4)

In [ ]:
# Reliability diagrams (win and draw classes) for the key models
def reliability(ax, P, y, k, label):
    p_k = P[:, k]
    bins = np.quantile(p_k, np.linspace(0, 1, 11))
    bins[0], bins[-1] = 0, 1
    idx = np.clip(np.digitize(p_k, bins) - 1, 0, 9)
    xs = [p_k[idx == b].mean() for b in range(10) if (idx == b).any()]
    ys = [(y[idx == b] == k).mean() for b in range(10) if (idx == b).any()]
    ax.plot(xs, ys, "o-", label=label, alpha=0.8)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for k, ax, title in [(2, axes[0], "P(win) calibration"), (1, axes[1], "P(draw) calibration")]:
    for name in ["Elo-only", "Ordinal LR", "Davidson-BT (MAP)", "Ensemble + temp"]:
        reliability(ax, test_probs[name], y_te, k, name)
    lim = ax.get_xlim()
    ax.plot([0, 1], [0, 1], "k--", lw=1)
    ax.set_xlim(lim)
    ax.set_xlabel("predicted probability"); ax.set_ylabel("observed frequency")
    ax.set_title(title); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Reading the results

Things to check when interpreting the table above:

- **Did anything beat Elo-only?** Expect gains to be real but small — Elo is itself a fitted predictor trained on millions of games; with ~2,000 training rows the headroom is mostly in draw modeling and calibration, not in re-learning skill.
- **Ordinal vs multinomial**: if the ordinal model matches or beats the multinomial at half the parameters, the proportional-odds restriction is supported. If multinomial wins, the draw class likely wants its own slope on $|d|$-type features.
- **GBT vs linear**: a meaningful gap would indicate nonlinear/interaction signal; parity confirms `diff_elo` dominance.
- **Davidson-BT**: its $\gamma$ converts to Elo points of first-move advantage (printed above) — sanity-check it against the ~52% white score from the EDA (≈ 15–20 Elo). Because unseen players fall back to priors, it degrades gracefully — a property the discriminative models lack.
- **Per-class recall on draws** will be ≈ 0 for argmax predictions everywhere — with a 7.8% base rate no calibrated model should ever *argmax* a draw. That is correct behavior for a probabilistic predictor, and exactly why log loss (not accuracy) is the primary metric.
- **Reliability diagrams**: temperature scaling should pull the ensemble onto the diagonal; Elo-only's flat draw probability shows as a horizontal cluster.